# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) and is loaded from:

```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare for record extraction.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# This is the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get the metadata (not subscripted to avoid accidental dict usage)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's explore the available record sets, their `@id`s, and fields.

In [ ]:
# List all record sets with their @id and name
print("Available record sets:")
for rset in dataset.record_sets():
    print(f"@id: {rset['@id']}, name: {rset.get('name', '[no name]')}")

In [ ]:
# For a demonstration, load the first record set
record_sets = list(dataset.record_sets())
if not record_sets:
    print('No record sets found in the dataset.')
else:
    main_record_set_id = record_sets[0]['@id']
    print(f"Fields in record set {main_record_set_id}:\n")
    for field in record_sets[0].get('field', []):
        print(f"  Field @id: {field['@id']}, name: {field.get('name', '[no name]')}, dataType: {field.get('dataType', '[unknown]')}")

## 3. Data Extraction
Load records from the main record set into a DataFrame for further analysis.

We'll use the record set's `@id` (variable: `main_record_set_id`) and reference fields by their `@id` as needed.

In [ ]:
dfs = dict()

if not record_sets:
    print("No record sets available for extraction.")
else:
    # You might want to extract others too, but we'll focus on the main survey records
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dfs[main_record_set_id] = df
    print(f"Loaded {len(df)} records. Columns available (with field @ids):\n")
    print(df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate common data processing by referencing fields via their `@id`.

Below, we select a numeric field (e.g., GAD-7 total score, identified by its field `@id` if available), filter, normalize, and group by another field.

In [ ]:
# Display all available field @ids to help user pick a field interactively
print("Columns in the main record set DataFrame:")
print(df.columns.tolist())

# We'll choose a likely field: GAD-7 total score (example name: 'gad7_total_score'), and group by 'village'
# Use actual @id if present
GAD7_FIELD_ID = 'gad7_total_score' if 'gad7_total_score' in df.columns else df.columns[0]
GROUP_FIELD_ID = 'village' if 'village' in df.columns else df.columns[1]

# Convert possible numeric
df[GAD7_FIELD_ID] = pd.to_numeric(df[GAD7_FIELD_ID], errors='coerce')

threshold = 10
filtered_df = df[df[GAD7_FIELD_ID] > threshold].copy()

print(f"\nFiltered records where {GAD7_FIELD_ID} > {threshold}:")
print(filtered_df[[GAD7_FIELD_ID, GROUP_FIELD_ID]].head())

filtered_df[f"{GAD7_FIELD_ID}_normalized"] = (filtered_df[GAD7_FIELD_ID] - filtered_df[GAD7_FIELD_ID].mean())/filtered_df[GAD7_FIELD_ID].std()
print(f"\nNormalized {GAD7_FIELD_ID} for filtered records (first 5):")
print(filtered_df[[GAD7_FIELD_ID, f"{GAD7_FIELD_ID}_normalized"]].head())

# Group by village and show average
if GROUP_FIELD_ID in filtered_df.columns:
    grouped_df = filtered_df.groupby(GROUP_FIELD_ID)[GAD7_FIELD_ID].mean().reset_index()
    print(f"\nAverage {GAD7_FIELD_ID} by {GROUP_FIELD_ID}:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the GAD-7 total score and compare means by village.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(8,5))
sns.histplot(df[GAD7_FIELD_ID].dropna(), kde=True, bins=15)
plt.title(f"Distribution of {GAD7_FIELD_ID}")
plt.xlabel(GAD7_FIELD_ID)
plt.show()

# Average score per group plot
if GROUP_FIELD_ID in df.columns:
    plt.figure(figsize=(10,5))
    avg_by_group = df.groupby(GROUP_FIELD_ID)[GAD7_FIELD_ID].mean().sort_values(ascending=False)
    avg_by_group.plot(kind='bar')
    plt.title(f"Average {GAD7_FIELD_ID} by {GROUP_FIELD_ID}")
    plt.ylabel(f"Mean {GAD7_FIELD_ID}")
    plt.xlabel(GROUP_FIELD_ID)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated loading, overview, and analysis of a Croissant-compliant open mental health survey dataset from Kenya using only public metadata and code.

**Key findings and steps:**
- We referenced dataset elements using their Croissant `@id` for reliability.
- Examined available record sets and fields.
- Filtered, normalized, and visualized scores (e.g., GAD-7) and explored variance by village.

Further work could involve detailed analysis with other fields, joining with additional metadata, or training predictive models using these survey features.
